# 🌾 Krishi-Veda Round 2: Decision-Tree Training
## Train SmolLM2-135M to combine NASA POWER + NPK + NDVI + Moon Phase → Farming Actions
**By Joydeep Das, Silchar, Assam**

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece torch

In [ ]:
import json, requests
# Load both datasets
url1 = 'https://raw.githubusercontent.com/divineearthly/Krishi-Veda-Module/main/training_data/vedic_farming_smollm2.json'
url2 = 'https://raw.githubusercontent.com/divineearthly/Krishi-Veda-Module/main/training_data/vedic_decision_training.json'
qa_data = requests.get(url1).json()
dec_data = requests.get(url2).json()
print(f'Loaded {len(qa_data)} Q&A pairs + {len(dec_data)} decision pairs')
# Convert decision pairs to chat format
for d in dec_data:
    qa_data.append({
        'messages': [
            {'role': 'user', 'content': f"Based on this data, what farming action should I take? {d['input']}"},
            {'role': 'assistant', 'content': d['output']}
        ]
    })
print(f'Total training examples: {len(qa_data)}')

In [ ]:
from datasets import Dataset
def format_chatml(ex):
    t = ''
    for m in ex['messages']:
        r = m['role']
        t += f"<|im_start|>{r}\n{m['content']}<|im_end|>\n"
    return {'text': t}
dataset = Dataset.from_list(qa_data).map(format_chatml)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(dataset["train"])}, Test: {len(dataset["test"])}')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
model_id = 'HuggingFaceTB/SmolLM2-135M-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id); tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map='auto')
print('Loaded SmolLM2-135M for Round 2 training!')

In [ ]:
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print('LoRA rank 16 for better decision learning')

In [ ]:
args = TrainingArguments(
    output_dir='./krishi-veda-round2',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=5,
    save_strategy='epoch',
    report_to='none',
    eval_strategy='epoch',
)
trainer = SFTTrainer(model=model, args=args, train_dataset=dataset['train'], eval_dataset=dataset['test'], tokenizer=tokenizer, max_seq_length=768, dataset_text_field='text')
print('Starting Round 2 training (5 epochs, 160 examples)...')
trainer.train()
print('Training complete!')

In [ ]:
# Test: General knowledge
pipe = __import__('transformers').pipeline('text-generation', model=model, tokenizer=tokenizer)
print('=== TEST 1: General Vedic Knowledge ===')
r = pipe("<|im_start|>user\nWhat is Panchgavya?<|im_end|>\n<|im_start|>assistant\n", max_new_tokens=80)
print(r[0]['generated_text'][:300])
print('\n=== TEST 2: Decision Making ===')
r2 = pipe("<|im_start|>user\nBased on this data, what farming action should I take? Temperature: 34°C, Rainfall: 10mm, Soil moisture: 40%, Crop: rice<|im_end|>\n<|im_start|>assistant\n", max_new_tokens=100)
print(r2[0]['generated_text'][:400])

In [ ]:
import torch
merged = model.merge_and_unload().to(torch.float16)
merged.save_pretrained('./krishi-veda-round2-merged')
tokenizer.save_pretrained('./krishi-veda-round2-merged')
print('Round 2 model saved in FP16!')
print('Download the krishi-veda-round2-merged folder')

## Convert to GGUF on your Android phone
```bash
cd ~ && unzip -o /storage/emulated/0/Download/krishi-veda-round2-merged.zip -d krishi-veda-round2
python3 ~/llama.cpp/convert_hf_to_gguf.py krishi-veda-round2 --outfile vedic-krishi-v2.gguf --outtype q8_0
~/llama.cpp/build/bin/llama-quantize vedic-krishi-v2.gguf vedic-krishi-v2-q4.gguf q4_k_m
~/llama.cpp/build/bin/llama-simple -m vedic-krishi-v2-q4.gguf -p "Temperature: 34°C, Rainfall: 10mm, what should I do?" 2>/dev/null
```